In [ ]:
!pip install -q langchain==0.0.219 openai==0.27.8

# !pip install -q transformers==4.29.2 sentencepiece==0.1.99 accelerate==0.19.0 bitsandbytes==0.39.0

!pip install -q python-dotenv==1.0.0

## OpenAI

In [ ]:
from dotenv import dotenv_values
env_values = dotenv_values("./app.env")

openai_api_key = env_values['OPEN_API_KEY']

In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(model_name="text-davinci-003", temperature=0.5, openai_api_key=openai_api_key)

### Generate Synonyms

In [ ]:
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate

In [ ]:
demo_message = """
Country Name: Egypt | Capital Name: Cairo
Country Name: Saudi Arabia | Capital Name: Riyadh
Country Name: France | Capital Name: Paris
Country Name: China |
""".strip()

print( llm( demo_message ) )

In [ ]:
demo_template = PromptTemplate(
    template="Country Name: {country} | Capital Name: {capital}",
    input_variables=['country', 'capital']
)

examples = [
    {"country": "Egypt", "capital": "Cairo"},
    {"country": "Saudi Arabia", "capital": "Riyadh"},
    {"country": "France", "capital": "Paris"},
]

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=demo_template,
    suffix="Country Name: {country} |",
    input_variables=['country']
)

In [ ]:
print( llm( prompt.format(country='Germany') ) )

## Riddles

In [ ]:
riddle_1 = """
I have five oranges, and the sum of what my sister and brother have is three times what I have plus thirty-five
""".strip()

print( llm( riddle_1 ) )

In [ ]:
riddle_2 = """
Total with what my family is 50 oranges. If we subtract one-fifth of the number from them, and add ten more oranges, how many oranges will there be in the end?
""".strip()

print( llm( riddle_2 ) )

In [ ]:
riddle_3 = """
When I was seven, my sister was twice my age. Now I am seventy years old, how old can my sister be?
""".strip()

print( llm( riddle_3 ) )

### Prepare some fewshot instructions

In [ ]:
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate

In [ ]:
examples = [
    {
        "question": "When I was seven, my sister was twice my age. Now I am seventy years old, how old can my sister be?",
        "answer": "\n".join([
            "We will followup some questions to get the answer.",
            "Follow up: How old was your sister when you were seven?",
            "Intermediate answer: Twice, which mean 14 years.",
            "Follow up: What is the difference between your age and your sister's age?",
            "Intermediate answer: 14 years - 7 years = 7 years.",
            "Follow up: When you were seventy years old, how old would your sister be?",
            "Intermediate answer: my age (70) +  The difference between me and my sister's age (7) = 77 years.",
            "Final Answer: 77 years."
        ])
    },
    {
        "question": "I have five oranges, and the sum of what my sister and brother have is three times what I have plus thirty-five?",
        "answer": "\n".join([
            "We will followup some questions to get the answer.",
            "Follow up: How many oranges do you have?",
            "Intermediate answer: 5 oranges.",
            "Follow up: How many oranges does your sister and brother have?",
            "Intermediate answer: three times: = 3 * 5 = 15. Also, we need to add thirty-five to them: 15 + 35 = 50 ",
            "Final Answer: 50 oranges."
        ])
    }
]

In [ ]:
example_prompt = PromptTemplate(input_variables=["question", "answer"], template="Question: {question}\n{answer}")

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}",
    input_variables=["input"]
)

message = prompt.format(input="Total with what my family is 50 oranges. If we subtract one-fifth of the number from them, and add ten more oranges, how many oranges will there be in the end?")
print( llm(message) )


We will followup some questions to get the answer.
Follow up: How many oranges does your family have?
Intermediate answer: 50 oranges.
Follow up: How many oranges will be subtracted?
Intermediate answer: One-fifth of the number: 50 / 5 = 10.
Follow up: How many oranges will be added?
Intermediate answer: Ten oranges.
Follow up: How many oranges will be left in the end?
Intermediate answer: 50 - 10 + 10 = 50 oranges.
Final Answer: 50 oranges.


### Generate Reviews

In [ ]:
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate

demo_prompt = PromptTemplate(template="Type for me 3 examples of a user complain about a product", input_variables=[])

demo_prompt_message = demo_prompt.format()

llm( demo_prompt_message )

'\n\n1. "I recently purchased your product and it arrived broken. I\'m very disappointed with the quality of this item."\n\n2. "I\'ve been using your product for a few weeks now and it seems to be malfunctioning. I\'m not happy with the performance of this product."\n\n3. "I\'ve been using your product for a few months now and it seems to be unreliable. I\'m not satisfied with the reliability of this product."'

In [ ]:
examples = [
    {
        "product": "mobile phone",
        "reviews": "\n- ".join([
                "I bought a new phone, and now, after two months, the screen does not work, The only thing customer service can do for you is waiting music",
                "I don't know what is going on with the battery of this device. I feel like I want to carry the charger wherever I go"
                ])
    },
    {
        "product": "vacuum cleaner",
        "reviews": "\n- ".join([
                "It has a voice beyond her capabilities",
                "Very innovative. You will always need to use a broom after using it",
            ])
    },
]

In [ ]:
example_prompt = PromptTemplate(input_variables=["product", "reviews"], template="Product: {product}.\nNegative and funny reviews:{reviews}.")

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Product: {product}",
    input_variables=["product"]
)

message = prompt.format(product="Chainsaw")
print( llm(message) )


Negative and funny reviews:This chainsaw is like a toddler trying to cut down a tree - it's slow and inefficient.
- I thought I was getting a powerful chainsaw, but instead I got a toy that makes a lot of noise.


### Generate Formatted Ingredients

In [ ]:
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate

demo_prompt = PromptTemplate(template="What're the ingredients to cook a pizza?", input_variables=[])

demo_prompt_message = demo_prompt.format()

llm( demo_prompt_message )

'\n\n1. Preheat oven to 425 degrees F (220 degrees C).\n\n2. Roll out pizza dough on a lightly floured surface.\n\n3. Spread a thin layer of pizza sauce over the dough.\n\n4. Sprinkle desired toppings over the sauce.\n\n5. Bake in preheated oven for 15 to 20 minutes, or until cheese is melted and crust is golden brown.\n\n6. Let pizza cool for a few minutes before slicing and serving.'

In [ ]:
examples = [
    {"recipe": "olives pizza", "ingredients": "\n".join([
                                    "* 1/2 pizza dough recipe",
                                    "* 1 tablespoon olive oil",
                                    "* 8 ounces cream cheese",
                                    "* 1/2 cup mozzarella cheese shredded",
                                    "* 1 tablespoon fresh parsley",
                                    "* 1/3 cup sliced green olives",
                                    "* 1/3 cup sliced black olives"
                                ])},
    {"recipe":"falafel", "ingredients": "\n".join(
                                [
                                    "* 1 cup dried chickpeas, soaked overnight",
                                    "* 1/2 cup onion",
                                    "* 1 cup parsley",
                                    "* 1 cup cilantro",
                                    "* 1 tablespoon fresh parsley",
                                    "* 1/3 cup sliced green olives",
                                    "* 1/3 cup sliced black olives",
                                    "* 3 garlic cloves",
                                    "* 1 tsp salt"
                                ]
                                )}
]

In [ ]:
example_prompt = PromptTemplate(
    template="Recipe Name: {recipe}\nIngredients: {ingredients}",
    input_variables=['recipe', 'ingredients']
)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Recipe Name: {recipe}\n",
    input_variables=['recipe']
)


In [ ]:
user_recipe = 'mashed potato'

message = prompt.format(recipe=user_recipe)

print( llm( message ) )